In [ ]:
#Shastha Rekha B :)
#The flow of this project is explained through comments i hope the spider team enjoys it.
# CELL 1 - note

# Spider ML Task 1 - Bonus Task
# Autoencoder on Fashion-MNIST using PyTorch
#
# What is an autoencoder?
# It is a neural network that learns to compress data and then reconstruct it.
# Encoder: compresses 784 pixels down to a small latent vector (e.g. 32 values)
# Decoder: takes that compressed vector and tries to recreate the original image
#
# Why is this useful?
# Dimensionality reduction, denoising, anomaly detection, generative modeling.
# The latent space captures the most important features of the data.
#
# I am experimenting with different latent dimensions to see how compression
# affects reconstruction quality.

In [ ]:
# CELL 2 - imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# CELL 3 - load data

# same Fashion-MNIST dataset as the base task
# for autoencoders I do not need labels during training
# the input image is both the input and the target

transform = transforms.Compose([
    transforms.ToTensor(),
    # normalizing to 0-1 range this time because I want to use Sigmoid at output
    # Sigmoid outputs values between 0 and 1, so labels should match
])

full_train_data = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

test_data = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_size = int(0.85 * len(full_train_data))
val_size = len(full_train_data) - train_size
train_dataset, val_dataset = random_split(full_train_data, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

print("Data loaded.")
print("Train:", train_size, "| Val:", val_size, "| Test:", len(test_data))

In [ ]:
# CELL 4 - define the autoencoder

# the architecture is fully connected (MLP-based) as asked in the task
# I am making the latent dimension a parameter so I can experiment with it easily

class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super(Autoencoder, self).__init__()

        # encoder squashes 784 -> 256 -> 128 -> latent_dim
        # each step compresses the information more
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )

        # decoder goes back up latent_dim -> 128 -> 256 -> 784
        # sigmoid at the end squashes output to 0-1 to match pixel values
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        # flatten the image first
        x = x.view(x.size(0), -1)

        # compress it
        latent = self.encoder(x)

        # reconstruct it
        reconstructed = self.decoder(latent)

        # reshape back to image format
        reconstructed = reconstructed.view(x.size(0), 1, 28, 28)
        return reconstructed

    def encode(self, x):
        # this lets me get just the compressed representation
        x = x.view(x.size(0), -1)
        return self.encoder(x)

    def decode(self, z):
        # this lets me decode from a latent vector
        out = self.decoder(z)
        return out.view(z.size(0), 1, 28, 28)

In [ ]:
# CELL 5 - training function

# I am writing this as a function so I can reuse it for different latent sizes

def train_autoencoder(latent_dim, num_epochs=25, learning_rate=0.001):
    print(f"\nTraining autoencoder with latent_dim = {latent_dim}")
    print("-" * 50)

    ae_model = Autoencoder(latent_dim=latent_dim).to(device)

    # MSE loss measures the average squared difference between original and reconstructed pixels
    # it is the standard choice for image reconstruction tasks
    reconstruction_loss = nn.MSELoss()

    ae_optimizer = optim.Adam(ae_model.parameters(), lr=learning_rate)
    ae_scheduler = optim.lr_scheduler.StepLR(ae_optimizer, step_size=10, gamma=0.5)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_weights = None

    for epoch in range(num_epochs):
        # training
        ae_model.train()
        total_train_loss = 0.0

        for images, _ in train_loader:
            images = images.to(device)

            ae_optimizer.zero_grad()

            # forward pass: try to reconstruct the image
            reconstructed = ae_model(images)

            # the target is the original image itself
            loss = reconstruction_loss(reconstructed, images)

            loss.backward()
            ae_optimizer.step()
            total_train_loss += loss.item()

        avg_train = total_train_loss / len(train_loader)

        # validation
        ae_model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)
                reconstructed = ae_model(images)
                loss = reconstruction_loss(reconstructed, images)
                total_val_loss += loss.item()

        avg_val = total_val_loss / len(val_loader)

        ae_scheduler.step()

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_weights = copy.deepcopy(ae_model.state_dict())

        train_losses.append(avg_train)
        val_losses.append(avg_val)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {avg_train:.5f} | Val Loss: {avg_val:.5f}")

    ae_model.load_state_dict(best_weights)
    print(f"Best val loss: {best_val_loss:.5f}")
    return ae_model, train_losses, val_losses, best_val_loss

In [ ]:
# CELL 6 - experiment with different latent dimensions

# this is the main experiment the task asks for
# I want to see how the size of the bottleneck affects reconstruction quality
# smaller latent_dim = more compression = harder to reconstruct well

latent_dims_to_try = [8, 16, 32, 64]
results = {}

for dim in latent_dims_to_try:
    trained_model, tr_losses, vl_losses, best_loss = train_autoencoder(
        latent_dim=dim, num_epochs=25
    )
    results[dim] = {
        'model': trained_model,
        'train_losses': tr_losses,
        'val_losses': vl_losses,
        'best_val_loss': best_loss
    }

print("\nSummary of experiments:")
print(f"{'Latent Dim':<15} {'Best Val Loss':<15}")
print("-" * 30)
for dim in latent_dims_to_try:
    print(f"{dim:<15} {results[dim]['best_val_loss']:.5f}")

In [ ]:
# CELL 7 - plot reconstruction loss for all experiments

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flat

for i, dim in enumerate(latent_dims_to_try):
    ax = axes[i]
    epochs_range = range(1, len(results[dim]['train_losses']) + 1)
    ax.plot(epochs_range, results[dim]['train_losses'], label='Train', color='steelblue')
    ax.plot(epochs_range, results[dim]['val_losses'], label='Val', color='tomato')
    ax.set_title(f'Latent Dim = {dim}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Autoencoder Reconstruction Loss for Different Latent Sizes', fontsize=13)
plt.tight_layout()
plt.savefig('autoencoder_loss_plots.png', dpi=150)
plt.show()

In [ ]:
# CELL 8 - visualize original vs reconstructed images

# this is the most important visualization for an autoencoder
# I want to see how well the model can reconstruct what it compressed

def show_reconstructions(model, data_loader, latent_dim, num_images=8):
    model.eval()
    images, labels = next(iter(data_loader))
    images_dev = images[:num_images].to(device)

    with torch.no_grad():
        reconstructed = model(images_dev)

    fig, axes = plt.subplots(2, num_images, figsize=(16, 4))

    for i in range(num_images):
        # original image on top row
        orig = images[i].squeeze().numpy()
        axes[0, i].imshow(orig, cmap='gray')
        axes[0, i].set_title('Original', fontsize=8)
        axes[0, i].axis('off')

        # reconstructed image on bottom row
        recon = reconstructed[i].squeeze().cpu().numpy()
        axes[1, i].imshow(recon, cmap='gray')
        axes[1, i].set_title('Recon', fontsize=8)
        axes[1, i].axis('off')

    plt.suptitle(f'Original vs Reconstructed (latent_dim = {latent_dim})', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'reconstruction_latent_{latent_dim}.png', dpi=150)
    plt.show()

# show reconstructions for each latent size
for dim in latent_dims_to_try:
    show_reconstructions(results[dim]['model'], val_loader, latent_dim=dim)

In [ ]:
# CELL 9 - side by side comparison of all latent sizes

# comparing the same 4 images reconstructed at each latent size
# this makes the effect of bottleneck size very clear

model_eval_imgs, _ = next(iter(val_loader))
model_eval_imgs = model_eval_imgs[:4]

fig, axes = plt.subplots(len(latent_dims_to_try) + 1, 4, figsize=(12, 14))

# first row is originals
for col in range(4):
    axes[0, col].imshow(model_eval_imgs[col].squeeze().numpy(), cmap='gray')
    axes[0, col].set_title('Original', fontsize=8)
    axes[0, col].axis('off')

# remaining rows are reconstructions for each latent dim
for row, dim in enumerate(latent_dims_to_try):
    model_to_use = results[dim]['model']
    model_to_use.eval()
    imgs_dev = model_eval_imgs.to(device)

    with torch.no_grad():
        recons = model_to_use(imgs_dev)

    for col in range(4):
        recon_img = recons[col].squeeze().cpu().numpy()
        axes[row + 1, col].imshow(recon_img, cmap='gray')
        axes[row + 1, col].set_title(f'Dim={dim}', fontsize=8)
        axes[row + 1, col].axis('off')

plt.suptitle('Effect of Latent Dimension on Reconstruction Quality', fontsize=12)
plt.tight_layout()
plt.savefig('latent_dim_comparison.png', dpi=150)
plt.show()

In [ ]:
# CELL 10 - visualize the latent space using t-SNE

# t-SNE is a dimensionality reduction technique that I use to visualize
# the 32-dimensional latent space in 2D
# if the autoencoder learned good representations, same-class items
# should cluster together in the latent space

from sklearn.manifold import TSNE

best_model = results[32]['model']  # using latent_dim=32 for this
best_model.eval()

all_latents = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        latent_vectors = best_model.encode(images)
        all_latents.append(latent_vectors.cpu().numpy())
        all_labels.extend(labels.numpy())

all_latents = np.concatenate(all_latents, axis=0)
all_labels = np.array(all_labels)

# only using 2000 samples because t-SNE is slow on large datasets
subset_size = 2000
idx = np.random.choice(len(all_latents), subset_size, replace=False)
latents_subset = all_latents[idx]
labels_subset = all_labels[idx]

print("Running t-SNE... this might take a minute")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
latents_2d = tsne.fit_transform(latents_subset)

class_names = [
    'T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

plt.figure(figsize=(12, 9))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for class_idx in range(10):
    mask = labels_subset == class_idx
    plt.scatter(
        latents_2d[mask, 0], latents_2d[mask, 1],
        c=[colors[class_idx]], label=class_names[class_idx],
        alpha=0.6, s=15
    )

plt.title('t-SNE of Autoencoder Latent Space (latent_dim=32)', fontsize=13)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('tsne_latent_space.png', dpi=150)
plt.show()

print("If clusters are visible, the autoencoder learned meaningful features.")

In [ ]:
# CELL 11 - save autoencoder weights

os.makedirs('saved_models', exist_ok=True)

for dim in latent_dims_to_try:
    model_to_save = results[dim]['model']
    save_data = {
        'model_state_dict': model_to_save.state_dict(),
        'latent_dim': dim,
        'best_val_loss': results[dim]['best_val_loss']
    }
    filename = f'saved_models/autoencoder_latent_{dim}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(save_data, f)
    print(f"Saved autoencoder with latent_dim={dim} to {filename}")

In [ ]:
# CELL 12 - denoising experiment (extra credit type thing)

# I want to see if the autoencoder can remove noise from images
# this is a common use case for autoencoders
# I add Gaussian noise to the input and see if the output is cleaner

best_ae = results[32]['model']
best_ae.eval()

noisy_batch, true_labels = next(iter(val_loader))
noisy_batch = noisy_batch[:8]

# adding random Gaussian noise to the images
noise_level = 0.3
noisy_input = noisy_batch + noise_level * torch.randn_like(noisy_batch)
noisy_input = torch.clamp(noisy_input, 0.0, 1.0)  # keep pixel values in valid range

with torch.no_grad():
    denoised = best_ae(noisy_input.to(device))

fig, axes = plt.subplots(3, 8, figsize=(18, 6))

for i in range(8):
    axes[0, i].imshow(noisy_batch[i].squeeze().numpy(), cmap='gray')
    axes[0, i].set_title('Original', fontsize=7)
    axes[0, i].axis('off')

    axes[1, i].imshow(noisy_input[i].squeeze().numpy(), cmap='gray')
    axes[1, i].set_title('Noisy', fontsize=7)
    axes[1, i].axis('off')

    axes[2, i].imshow(denoised[i].squeeze().cpu().numpy(), cmap='gray')
    axes[2, i].set_title('Denoised', fontsize=7)
    axes[2, i].axis('off')

plt.suptitle('Denoising with Autoencoder (noise=0.3)', fontsize=12)
plt.tight_layout()
plt.savefig('denoising_result.png', dpi=150)
plt.show()

In [ ]:
# CELL 13 - summary of findings

print("Summary of Autoencoder Experiments")
print("=" * 45)
print()
print("Reconstruction loss (MSE) by latent dimension:")
for dim in latent_dims_to_try:
    print(f"  latent_dim = {dim:3d}  ->  val loss = {results[dim]['best_val_loss']:.5f}")
print()
print("Observations:")
print("- Larger latent dims give better reconstruction because they store more info")
print("- Smaller latent dims produce blurrier reconstructions but compress more")
print("- t-SNE shows that the latent space forms rough class clusters")
print("- The autoencoder can partially denoise images even without denoising training")
print()
print("Possible improvements:")
print("- Train as a denoising autoencoder by adding noise during training")
print("- Use convolutional layers (CNN) instead of MLP for better spatial features")
print("- Try a Variational Autoencoder (VAE) to get a proper generative model")
print("- Use the encoder as feature extractor for classification (transfer learning)")